In [ ]:
%pip install torch
%pip install transformers
%pip install datasets
%pip install peft
%pip install bitsandbytes
%pip install accelerate
%pip install evaluate

In [ ]:
#imports
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset, concatenate_datasets

In [ ]:
#Global Variables
LORA = True
QUANTIZE = False
CLUSTER = True
CUDA = True
CPU = False
MODEL = "Qwen/Qwen3-8B"

In [ ]:
device = torch.device("cuda" if CUDA and torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
tasks = [
    "narrativeqa", "qasper", "multifieldqa_en", "multifieldqa_zh", 
    "hotpotqa", "2wikimqa", "musique", "dureader", "gov_report", 
    "qmsum", "multi_news", "vcsum", "trec", "triviaqa", "samsum",
    "lsht", "passage_count", "passage_retrieval_en", "passage_retrieval_zh",
    "lcc", "repobench-p"
]
datasets = {}
for task in tasks:
    if CLUSTER:
        datasets[task] = load_dataset("THUDM/LongBench", task, split="test", cache_dir='/home/rhayrapetyan/.cache/huggingface/datasets/downloads/extracted/0fe2e4841dad9538f66462ef845f9ebb8d619462c2028d55ba8593b7c60267f6/data')
    else:
        datasets[task] = load_dataset("THUDM/LongBench", task, split="test")

In [ ]:
#dataset generation
def format(training_example):
    prompt = (
        "Given the following context, answer the question concisely.\n\n"
        f"Question: {training_example['input']}\n"
        f"Context: {training_example['context']}\n\n"
    )
    answer = training_example["answers"]

    if isinstance(answer, list):
        answer = answer[0]
    elif answer == [""] or "":
        raise ValueError("Empty answer found in the dataset.")

    return {"prompt": prompt, "response": answer}

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(training_example):
    prompt_tokens = tokenizer(training_example["prompt"])['input_ids']
    answer_tokens = tokenizer(training_example["response"])['input_ids']
    input_ids = prompt_tokens + answer_tokens + [tokenizer.eos_token_id]
    labels = [-100] * len(prompt_tokens) + answer_tokens + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

In [ ]:
train_datasets = {}
for task_name, ds in datasets.items():
    train_datasets[task_name] = ds.map(format, remove_columns=ds.column_names)
all_train_dataset = concatenate_datasets(list(train_datasets.values()))
tokenized_dataset = all_train_dataset.map(tokenize, remove_columns=all_train_dataset.column_names)

In [ ]:
train, test = tokenized_dataset.train_test_split(test_size=0.1).values()

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb

model = None
print("Loading model...")

model = None
bnb_config = None

if QUANTIZE:
    print("  Applying 4-bit quantization config...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config if QUANTIZE else None,
    device_map="auto" if CUDA else None,
    trust_remote_code=True
)

if LORA:
    print("  Adding LoRA adapters...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base_model, lora_config)
else:
    model = base_model

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

if not QUANTIZE:
    model.to(device)

print("Model ready ✅\n")
print(model)



Loading full-precision model for LoRA...


NameError: name 'CUDA' is not defined

In [ ]:
from evaluate import load

metric = load("rouge")

def compute_metrics(predictions, references):
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_refs = tokenizer.batch_decode(references, skip_special_tokens=True)
    result = metric.compute(predictions=decoded_preds, references=decoded_refs, use_stemmer=True)
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    return result

In [ ]:
from transformers import TrainingArguments

mode = []
if QUANTIZE and LORA:
    mode.append("quantized_lora")
if QUANTIZE:
    mode.append("quantized")
if LORA:
    mode.append("lora")
if not mode:
    mode.append("fp")  # full precision, no quantization, no LoRA

output_dir = f"./Qwen3-8B-LongBench-finetuned-{'_'.join(mode)}"

training_args = TrainingArguments(
    output_dir=output_dir,
    do_eval=True,
    eval_strategy="steps",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    save_steps=600,
    eval_steps=300,
    logging_steps=300,
    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,
    push_to_hub=False,
    use_cpu=CPU
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=lambda data: {
        'input_ids': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['input_ids']) for f in data], batch_first=True, padding_value=tokenizer.pad_token_id
        ),
        'attention_mask': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['attention_mask']) for f in data], batch_first=True, padding_value=0
        ),
        'labels': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['labels']) for f in data], batch_first=True, padding_value=-100
        ),
    },
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=tokenizer
)



In [ ]:
trainer.train()